# Silver Layer

## Silver Crypto List

In [0]:
-- ─────────────────────────────────────────
-- silver_crypto_list
-- Source  : crypto_exchange.bronze.structured_crypto_list
-- Dedup   : latest record per symbol
-- Drops   : raw ingestion metadata (year, month, day, input_file_path, ingest_timestamp)
-- ─────────────────────────────────────────

CREATE OR REFRESH MATERIALIZED VIEW crypto_exchange.silver.silver_crypto_list
COMMENT "Silver — deduplicated crypto reference list, one row per coin (latest ingestion)"
AS
SELECT
  id,
  symbol,
  name,
  source_exchange,
  history_available_from,
  ohlc_available_from,
  ingestion_timestamp,
  api_source
FROM crypto_exchange.bronze.structured_crypto_list
QUALIFY ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY ingestion_timestamp DESC) = 1

## Silver Conversions

In [0]:
-- ─────────────────────────────────────────
-- silver_conversions
-- Source  : crypto_exchange.bronze.structured_conversions
-- Dedup   : one row per (from_symbol, to_symbol, ingestion_timestamp) — all history preserved
-- Derived : pair label, inverse_rate
-- Drops   : raw ingestion metadata
-- ─────────────────────────────────────────

CREATE OR REFRESH MATERIALIZED VIEW crypto_exchange.silver.silver_conversions
COMMENT "Silver — full conversion rate history per currency pair with derived inverse rate"
AS
SELECT
  from_symbol,
  to_symbol,
  CONCAT(from_symbol, '_', to_symbol)  AS pair,
  amount,
  converted_amount,
  rate,
  ROUND(1.0 / rate, 8)                AS inverse_rate,
  ingestion_timestamp,
  api_source
FROM crypto_exchange.bronze.structured_conversions
QUALIFY ROW_NUMBER() OVER (PARTITION BY from_symbol, to_symbol, ingestion_timestamp ORDER BY ingestion_timestamp DESC) = 1

## Silver Market Data

In [0]:
-- ─────────────────────────────────────────
-- silver_market_data
-- Source  : crypto_exchange.bronze.structured_market_data
--           LEFT JOIN crypto_exchange.silver.silver_crypto_list
-- Dedup   : latest record per (symbol, price_timestamp)
-- Enrich  : coin_id, coin_name from silver_crypto_list
-- Derived : price_change_direction, market_cap_tier, price_position
-- Drops   : raw ingestion metadata
-- ─────────────────────────────────────────

CREATE OR REFRESH MATERIALIZED VIEW crypto_exchange.silver.silver_market_data
COMMENT "Silver — deduplicated market data enriched with coin metadata and derived KPIs"
AS
SELECT
  m.symbol,
  cl.id                                                              AS coin_id,
  cl.name                                                            AS coin_name,
  m.price,
  m.highest,
  m.lowest,
  m.change_24h,

  -- Where the current price sits within today's range (0.0 = at daily low, 1.0 = at daily high)
  ROUND(
    (m.price - m.lowest) / NULLIF(m.highest - m.lowest, 0),
    4
  )                                                                  AS price_position,

  m.market_cap,
  m.volume,
  m.source_exchange,

  -- 24-hour price movement direction
  CASE
    WHEN m.change_24h > 0 THEN 'UP'
    WHEN m.change_24h < 0 THEN 'DOWN'
    WHEN m.change_24h = 0 THEN 'FLAT'
    ELSE NULL
  END                                                                AS price_change_direction,

  -- Market cap classification (thresholds in USD)
  CASE
    WHEN m.market_cap >= 10000000000 THEN 'LARGE_CAP'   -- >= $10B
    WHEN m.market_cap >= 1000000000  THEN 'MID_CAP'    -- $1B – $10B
    WHEN m.market_cap IS NOT NULL    THEN 'SMALL_CAP'  -- < $1B
    ELSE NULL
  END                                                                AS market_cap_tier,

  m.price_timestamp,
  m.ingestion_timestamp,
  m.api_source

FROM crypto_exchange.bronze.structured_market_data m
LEFT JOIN crypto_exchange.silver.silver_crypto_list cl
  ON m.symbol = cl.symbol
QUALIFY ROW_NUMBER() OVER (PARTITION BY m.symbol, m.price_timestamp ORDER BY m.ingestion_timestamp DESC) = 1